In [2]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error

SMILES_COL = "smiles"
LABEL_COL = "gap_eV"
N_SPLITS = 5
RANDOM_STATE = 42

FEATURE_FILES = {
    "descriptors": "../train/qm9_descriptors.csv",
    "morgan": "../train/qm9_morgan_fp.csv",
    "maccs": "../train/qm9_maccs_keys.csv",
}

# データ読み込み
train_dfs = {name: pd.read_csv(path) for name, path in FEATURE_FILES.items()}
y_train = train_dfs["descriptors"][LABEL_COL].values
n_train = len(y_train)

kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

oof_preds = {name: np.zeros(n_train) for name in FEATURE_FILES}
base_models = {name: [] for name in FEATURE_FILES}  # 各foldのモデルを保存(val評価用)

for name, df in train_dfs.items():
    X = df.drop(columns=[SMILES_COL, LABEL_COL])
    y = df[LABEL_COL].values

    for fold, (train_idx, valid_idx) in enumerate(kf.split(X), start=1):
        X_tr, X_val = X.iloc[train_idx], X.iloc[valid_idx]
        y_tr, y_val = y[train_idx], y[valid_idx]

        model = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, random_state=RANDOM_STATE)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(50, verbose=False)],
        )

        oof_preds[name][valid_idx] = model.predict(X_val)
        base_models[name].append(model)

    fold_mae = mean_absolute_error(y, oof_preds[name])
    print(f"{name}: OOF MAE = {fold_mae:.4f}")

# メタモデル用の特徴量(3モデルのOOF予測値)
meta_X_train = pd.DataFrame(oof_preds)
meta_y_train = y_train

meta_model = Ridge(alpha=1.0)
meta_model.fit(meta_X_train, meta_y_train)

stacked_pred_train = meta_model.predict(meta_X_train)
stacked_mae_train = mean_absolute_error(meta_y_train, stacked_pred_train)
print(f"\nスタッキング(train, OOF基準) MAE = {stacked_mae_train:.4f}")
print(f"メタモデルの重み: {dict(zip(meta_X_train.columns, meta_model.coef_))}")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.015273 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 21129
[LightGBM] [Info] Number of data points in the train set: 85632, number of used features: 184
[LightGBM] [Info] Start training from score 6.833426
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.015029 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 21133
[LightGBM] [Info] Number of data points in the train set: 85632, number of used features: 184
[LightGBM] [Info] Start training from score 6.834964
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.013359 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not e

KeyboardInterrupt: 

In [ ]:
VAL_FEATURE_FILES = {
    "descriptors": "val/qm9_descriptors.csv",
    "morgan": "val/qm9_morgan_fp.csv",
    "maccs": "val/qm9_maccs_keys.csv",
}

val_dfs = {name: pd.read_csv(path) for name, path in VAL_FEATURE_FILES.items()}
y_val = val_dfs["descriptors"][LABEL_COL].values

# 各特徴量タイプごとに、5foldぶんのモデルの予測を平均してval予測とする
val_preds = {}
for name, df in val_dfs.items():
    X_val = df.drop(columns=[SMILES_COL, LABEL_COL])
    fold_preds = np.column_stack([m.predict(X_val) for m in base_models[name]])
    val_preds[name] = fold_preds.mean(axis=1)

meta_X_val = pd.DataFrame(val_preds)
stacked_pred_val = meta_model.predict(meta_X_val)
stacked_mae_val = mean_absolute_error(y_val, stacked_pred_val)

print(f"\nスタッキング(val) MAE = {stacked_mae_val:.4f}")
for name in FEATURE_FILES:
    single_mae = mean_absolute_error(y_val, val_preds[name])
    print(f"  {name}単体(val) MAE = {single_mae:.4f}")

NameError: name 'pd' is not defined

In [ ]:
import japanize_matplotlib
import matplotlib.pyplot as plt

# 1. train(OOF)でのMAE比較（各モデル単体 vs スタッキング）
model_names = list(FEATURE_FILES.keys()) + ["stacking"]
train_maes = [mean_absolute_error(y_train, oof_preds[name]) for name in FEATURE_FILES]
train_maes.append(stacked_mae_train)

plt.figure(figsize=(6, 4))
bars = plt.bar(model_names, train_maes, color=["steelblue"] * 3 + ["darkorange"])
plt.ylabel("MAE (eV)")
plt.title("train(OOF)でのMAE比較")
for bar, mae in zip(bars, train_maes):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f"{mae:.4f}", ha="center", va="bottom")
plt.tight_layout()
plt.savefig("results/stacking_train_mae_comparison.png", dpi=150)
plt.show()

# 2. メタモデルの重み（各ベースモデルをどれくらい信用しているか）
plt.figure(figsize=(6, 4))
plt.bar(meta_X_train.columns, meta_model.coef_, color="mediumseagreen")
plt.axhline(0, color="gray", linewidth=0.8)
plt.ylabel("メタモデルの係数(重み)")
plt.title("各ベースモデルへの重み")
plt.tight_layout()
plt.savefig("results/stacking_meta_weights.png", dpi=150)
plt.show()

# 3. スタッキング予測 vs 実測（train, OOF基準）
plt.figure(figsize=(6, 6))
plt.scatter(y_train, stacked_pred_train, s=3, alpha=0.3)
lims = [y_train.min(), y_train.max()]
plt.plot(lims, lims, color="red", linestyle="--")
plt.xlabel("実測 gap_eV")
plt.ylabel("予測 gap_eV (スタッキング)")
plt.title(f"スタッキング予測 vs 実測 (train, MAE={stacked_mae_train:.4f})")
plt.tight_layout()
plt.savefig("results/stacking_pred_vs_true.png", dpi=150)
plt.show()

In [ ]:
descriptors_mae = mean_absolute_error(y_train, oof_preds["descriptors"])
improvement = descriptors_mae - stacked_mae_train
improvement_pct = improvement / descriptors_mae * 100

print(f"記述子単体のMAE:     {descriptors_mae:.4f} eV")
print(f"スタッキングのMAE:   {stacked_mae_train:.4f} eV")
print(f"改善幅:              {improvement:.4f} eV ({improvement_pct:+.2f}%)")

if improvement_pct > 1:
    print("→ スタッキングは意味のある改善をしている")
else:
    print("→ スタッキングの効果はごくわずか。記述子単体でほぼ同等の精度")

In [ ]:
import japanize_matplotlib
import matplotlib.pyplot as plt

labels = ["記述子単体", "スタッキング"]
values = [descriptors_mae, stacked_mae_train]

plt.figure(figsize=(5, 4))
bars = plt.bar(labels, values, color=["steelblue", "darkorange"])
plt.ylabel("MAE (eV)")
plt.title("記述子単体 vs スタッキング")
for bar, v in zip(bars, values):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f"{v:.4f}", ha="center", va="bottom")
plt.tight_layout()
plt.savefig("results/descriptors_vs_stacking.png", dpi=150)
plt.show()｀